## Impact of Program Design

A program that runs is not a program that works. One passing test says nothing about the input you didn't imagine -- and once code ships, it affects real people in ways its author never intended.

### Learning Targets

- I can explain system reliability and why one passing test isn't evidence of it.
- I can add validation that rejects invalid input before it corrupts an object.
- I can describe beneficial and harmful effects of the same program, including harm beyond its intended use.
- I can decide whether online code may be reused, based on its license.

## Lesson Design: The LxD Cycle

This lesson was built with the Learning Experience Design cycle -- every choice below traces to something observed about actual learners.

<div style="height:2rem"></div>

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart LR
    E("<b>Empathize</b><br/>Students test one input,<br/>then move on"):::start --> D("<b>Define</b><br/>A method owns its<br/>invalid input"):::ask
    D --> I("<b>Ideate</b><br/>Trace table with<br/>a wrong row"):::good
    I --> P("<b>Prototype &amp; Test</b><br/>Worked example<br/>+ practice"):::info
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

<div style="height:2rem"></div>

### Empathize

Watching classmates write their first validated methods, the same pattern showed up repeatedly: they tested one input, it returned the expected value, and they moved on. When asked "what happens if someone passes a negative number," the common answer was some version of *"why would they?"*

The misconception underneath it: **invalid input is something users do wrong, not something the method is responsible for handling.** Testing is treated as confirmation that the code works, not as an attempt to break it.

This is worth teaching because it produces no error. The code compiles, one test passes, and the failure surfaces later in a context where nobody is looking for it.

### Define

**Point of View:** A CSA student who can write a working method needs to see that a method owns its own invalid input -- they currently test to confirm success, not to find failure, which hides silent corruption.

### Ideate

**How Might We:** make a silent failure *visible* to someone whose code has never errored? Chosen activity: a trace table with a deliberately wrong row, so students watch a method accept -400 degrees and confidently report a status. `Thermostat` was chosen because an impossible value needs no domain knowledge to spot, and it scales into the impact sections below.

### Prototype & Test

The worked example, the broken-then-fixed pair, and the practice tasks below are that prototype -- taught and refined on the assigned teaching day.

## The Running Example

**ClimateSense** is a hypothetical connected thermostat installed in tens of thousands of homes. Everything below centers on its `Thermostat` class, which holds a target temperature and reports the system's status:

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart LR
    T(["targetTemp"]):::start --> A{"&ge; 78 ?"}:::ask
    A -- yes --> C(["COOLING"]):::info
    A -- no --> B{"&le; 65 ?"}:::ask
    B -- yes --> H(["HEATING"]):::bad
    B -- no --> I(["IDLE"]):::good
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

## Part 1: System Reliability

**System reliability** means a program performs as expected, under stated conditions, without failure -- not just for the one input you tried. Here is ClimateSense's unguarded first version. Predict all three outputs before reading on.

In [ ]:
// CODE_RUNNER: Predict all three outputs first, then press Run -- does anything look wrong?
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        System.out.println(t.getStatus());

        t.setTargetTemp(80);
        System.out.println(t.getStatus());

        t.setTargetTemp(-400);          // below absolute zero
        System.out.println(t.getStatus());
    }
}
Thermostat.main(null);

| Call | `targetTemp` after | `getStatus()` | Correct? |
|---|---|---|---|
| `new Thermostat("Lab", 72)` | 72.0 | `IDLE` | Yes |
| `setTargetTemp(80)` | 80.0 | `COOLING` | Yes |
| `setTargetTemp(-400)` | -400.0 | `HEATING` | **No** -- below absolute zero |

No crash, no warning -- just a confident wrong answer. A silent failure is worse than a crash, because a crash tells you where to look.

### Popcorn Hacks

1. What does `getStatus()` return for exactly 78? For exactly 65?

In [ ]:
// CODE_RUNNER: Popcorn Hack 1 -- Press Run and read the status for exactly 78 and exactly 65.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        t.setTargetTemp(78);
        System.out.println("78 -> " + t.getStatus());
        t.setTargetTemp(65);
        System.out.println("65 -> " + t.getStatus());
    }
}
Thermostat.main(null);

## Part 2: Guarding Against Bad Input

A **guard clause** sits at the top of a method and rejects invalid input *before* anything else happens. The first version had none, so `-400`, `9999`, and `Double.NaN` all succeeded silently.

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart LR
    S("setTargetTemp(temp)"):::start --> V{"NaN, or outside<br/>50 - 90 ?"}:::ask
    V -- yes --> X("throw IllegalArgumentException<br/><b>object unchanged</b>"):::bad
    V -- no --> OK("targetTemp = temp<br/><b>value stored</b>"):::good
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

In [ ]:
// CODE_RUNNER: Press Run to see the guard reject -400 and leave the object unchanged.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(-400);
        } catch (IllegalArgumentException e) {
            System.out.println("rejected -> " + e.getMessage());
        }
        System.out.println(t.getTargetTemp());   // still 72.0 -- the exception ran before the assignment
    }
}
Thermostat.main(null);

A rejected call leaves the object exactly as it was -- failing is safer than corrupting. `NaN` needs its own check because it fails every comparison: `NaN < 50` and `NaN > 90` are both `false`, so a range check alone lets it through.

### Popcorn Hacks

1. Delete the `Double.isNaN(temp)` clause and predict what `setTargetTemp(Double.NaN)` does. Explain in one sentence.

In [ ]:
// CODE_RUNNER: Popcorn Hack 2 -- Press Run, then delete the Double.isNaN(temp) clause and run again. What changes?
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(Double.NaN);
            System.out.println("accepted -> " + t.getTargetTemp());
        } catch (IllegalArgumentException e) {
            System.out.println("rejected -> " + e.getMessage());
        }
    }
}
Thermostat.main(null);

## Part 3: Social, Economic, and Cultural Impact

One program can be beneficial and harmful at once -- the harm doesn't cancel the benefit. ClimateSense in fifty thousand homes:

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart LR
    C("<b>ClimateSense</b><br/>50,000 homes"):::start
    C --> E("Energy use"):::info
    C --> S("Auto scheduling"):::info
    C --> R("Remote access"):::info
    E --> E1("Lower bills,<br/>less grid strain"):::good
    S --> S1("Comfort without<br/>manual effort"):::good
    S --> S2("Penalizes<br/>shift workers"):::bad
    R --> R1("Adjust before<br/>arriving"):::good
    R --> R2("Needs smartphone<br/>+ home internet"):::bad
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

None of the harms are bugs -- they're consequences of reasonable design choices. Ask who's *excluded*, not just who's served.

## Part 4: Unintended Consequences

An **unintended consequence** is harm beyond a program's intended use, even when the code works exactly as designed. Suppose ClimateSense raises every target 2 degrees during a heat wave to ease grid load:

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart LR
    A("<b>+2 degrees</b><br/>in one house"):::start --> B("Barely noticeable"):::good
    A --> C("x 50,000 houses at once<br/><b>demand shock on the grid</b>"):::bad
    A --> H("Medically vulnerable resident<br/><b>real health risk</b>"):::bad
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

Nobody wrote a harmful feature. The harm comes from **scale**, not intent.

## Part 5: Intellectual Property and Code Reuse

Reusing code is normal -- the rules are about *which* code, under *what terms*.

<pre class="mermaid">
%%{init: {'theme':'base','fontFamily':'JetBrains Mono, monospace','themeVariables':{'fontFamily':'JetBrains Mono, monospace','fontSize':'14px','primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#60a5fa','lineColor':'#94a3b8','edgeLabelBackground':'#0f172a'},'flowchart':{'curve':'basis','padding':16,'nodeSpacing':40,'rankSpacing':50}}}%%
flowchart TD
    Q("Found code online"):::start --> L{"License stated?"}:::ask
    L -- no --> N("Treat as NOT free to use<br/><b>write your own</b>"):::bad
    L -- yes --> O{"Open source?"}:::ask
    O -- yes --> Y("Use it,<br/><b>follow the license</b>"):::good
    O -- no --> P("<b>Only with permission</b>"):::bad
    classDef start fill:#1e3a8a,stroke:#60a5fa,color:#fff,stroke-width:2px
    classDef ask fill:#78350f,stroke:#fbbf24,color:#fff,stroke-width:2px
    classDef good fill:#14532d,stroke:#4ade80,color:#fff,stroke-width:2px
    classDef bad fill:#7f1d1d,stroke:#f87171,color:#fff,stroke-width:2px
    classDef info fill:#1e293b,stroke:#94a3b8,color:#f8fafc,stroke-width:2px
</pre>

**Readable is not reusable.** A public repo is visible to everyone; that says nothing about permission.

### Popcorn Hacks

1. A classmate says a four-line snippet is too short for licensing to apply. Respond.

In [ ]:
// CODE_RUNNER: Popcorn Hack 3 -- A classmate says a four-line snippet is too short for licensing. Run it, then change license to "MIT" and "proprietary" and compare.
public class LicenseCheck {
    static String mayReuse(String license) {
        if (license.equals("none")) return "NO -- no license means no permission. Write your own.";
        if (license.equals("MIT") || license.equals("Apache-2.0")) return "YES -- follow the license (keep the notice).";
        if (license.equals("proprietary")) return "ONLY with the owner's permission.";
        return "Unknown license -- read it before reusing.";
    }

    public static void main(String[] args) {
        String snippet = "int add(int a, int b) { return a + b; }";   // short, but still someone's work
        String license = "none";                                      // try "MIT", "proprietary"

        System.out.println("Snippet: " + snippet);
        System.out.println("License: " + license);
        System.out.println("May I reuse it? " + mayReuse(license));
    }
}
LicenseCheck.main(null);

## Practice: Trace and Debug

### Predict the Output

Using the **guarded** `setTargetTemp`:

In [ ]:
// CODE_RUNNER: Pick your answer (A-D) below first, then press Run to check it.
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }

    public static void main(String[] args) {
        Thermostat t = new Thermostat("Lab", 72);
        try {
            t.setTargetTemp(95);
        } catch (IllegalArgumentException e) {
            System.out.println("rejected");
        }
        System.out.println(t.getTargetTemp());
        System.out.println(t.getStatus());
    }
}
Thermostat.main(null);

A. `95.0`, then `COOLING`

B. `rejected`, then `72.0`, then `IDLE`

C. `rejected`, then `95.0`, then `COOLING`

D. `rejected`, then `72.0`, then `COOLING`

### Popcorn Hacks: Apply the Idea

Finish `setRoomName` so it rejects `null`, empty, and whitespace-only names -- validate before assigning. The first attempt should be accepted and the next three rejected.

In [ ]:
// CODE_RUNNER: Popcorn Hack 4 -- Finish setRoomName so it rejects null, empty, and whitespace-only names, then press Run. Expect 1 accepted, 3 rejected.
public class Room {
    private String roomName;

    public Room(String roomName) { this.roomName = roomName; }

    public void setRoomName(String name) {
        // TODO: reject null, empty, and whitespace-only names.
        roomName = name;
    }

    public String getRoomName() { return roomName; }

    public static void main(String[] args) {
        Room r = new Room("Lab");
        String[] attempts = { "Nursery", null, "", "   " };
        for (String a : attempts) {
            try {
                r.setRoomName(a);
                System.out.println("accepted -> [" + r.getRoomName() + "]");
            } catch (IllegalArgumentException e) {
                System.out.println("rejected -> " + e.getMessage());
            }
        }
    }
}
Room.main(null);

## Answer Check

**Predict the Output -- B.** 95 is outside 50-90, so the guard throws before the assignment; the object keeps 72.0 and `getStatus()` returns `IDLE`. (A and C assume the value was stored anyway; D contradicts its own state.)

**Apply the Idea.** Reject `null` (a later method call on it would throw `NullPointerException` far from the real cause) and empty or whitespace-only names (an unidentifiable room is unusable). The rule: a mutator rejects anything that leaves the object unable to function.

## Quick Review

- **Reliability:** works under all stated conditions, not just the one input you tried.
- **Test edges:** zero, negative, empty, `null`, `NaN`, boundaries.
- **Guard clause:** runs before assignment, so invalid input is never stored.
- **Silent acceptance** is more dangerous than a crash.
- **Impact:** one program can help and harm, and harm beyond intended use comes from scale, not intent.
- **Reuse:** visible code isn't free code. No license means no permission.

## Sources

- College Board, *AP Computer Science A Course and Exam Description*, Unit 3, Topic 3.2.
- Oracle, ["Class Double"](https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/Double.html) -- `isNaN` and `NaN` comparison behavior.
- Oracle, ["Class IllegalArgumentException"](https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/IllegalArgumentException.html).
- Open Source Initiative, ["The Open Source Definition"](https://opensource.org/osd).
- GitHub Docs, ["Licensing a repository"](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/licensing-a-repository).